# EfficientNet-B3 Brain Tumour Classification (4-class) + Grad-CAM

**Stage 0b of the ORACLE pipeline.** Trains an ImageNet-pretrained EfficientNet-B3 to
classify a brain MRI slice as **glioma / meningioma / notumor / pituitary**, and explains
each prediction with **Grad-CAM**.

- **Dataset**: [`masoudnickparvar/brain-tumor-mri-dataset`](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) — ~7 023 JPEGs in `Training/` + `Testing/`, four class folders each.
- **Backbone**: `torchvision.models.efficientnet_b3`, ImageNet transfer, 3-phase unfreeze schedule.
- **Deployed weights are the EMA**, selected on macro-F1 — same convention as `best_nnunet2d.pth`.
- **Outputs**: `best_effnetb3_cls.pth` (bare EMA state_dict) and `effnetb3_cls_final.pt` (rich checkpoint + `load_classifier()`).

### Two caveats that belong up front

> **1 · The test split is optimistic.** This dataset aggregates three public sources
> (figshare, SARTAJ, Br35H). Its official `Testing/` split is **not patient-disjoint** from
> `Training/`, and near-duplicate images exist across the two. §3 counts the exact
> cross-split duplicate hashes. Read every number in §10 as an **upper bound**, not a
> generalisation estimate.
>
> **2 · Using this model inside the ORACLE pipeline is out-of-distribution.** These are
> *pre-operative* 2D JPEGs across mixed scanners and planes. MU-Glioma-Post — what
> `full_pipeline_testing.ipynb` runs on — is **100 % glioma and post-operative**. The
> classifier's prediction there is a pipeline-wiring demonstration with **no diagnostic
> weight**, and is flagged as such in the notebook, in `metrics.json`
> (`out_of_distribution: true`), and in the 3D viewer.

### Kaggle setup

Add data → search `brain-tumor-mri-dataset` → Add. **Turn Internet ON**
(Settings → Internet) so torchvision can fetch ImageNet weights — without them the model
trains from random init and still reports plausible-looking accuracy. §7 records a
`pretrained` flag in the checkpoint and §13 refuses to declare success if it is `False`.


## 1. Setup & Imports

In [ ]:
import os, re, glob, json, math, copy, random, hashlib, time, datetime, zipfile, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

# Keep downloaded ImageNet weights inside the notebook OUTPUT so the run can be
# republished as a Kaggle dataset and every later run works with Internet OFF.
if Path('/kaggle/working').is_dir():
    torch.hub.set_dir('/kaggle/working/torch_hub')

print('device       :', DEVICE)
print('torch        :', torch.__version__)
import torchvision; print('torchvision  :', torchvision.__version__)
print('albumentations:', A.__version__)   # 2.x renamed several kwargs — see section 5

## 2. Configuration

In [ ]:
# ── Frozen contract shared with full_pipeline_testing.ipynb ───────────────
# These exact keys/values are re-declared in the pipeline notebook so the
# checkpoint loads there with strict=True. Do not reorder `classes`.
CLS_CFG_DEFAULTS = {
    'classes':    ['glioma', 'meningioma', 'notumor', 'pituitary'],   # frozen, alphabetical
    'img_size':   300,          # B3 native resolution — NOT the segmenter's 256
    'dropout':    0.3,
    'norm_mean':  [0.485, 0.456, 0.406],   # ImageNet — required by the pretrained backbone
    'norm_std':   [0.229, 0.224, 0.225],
    'denoise':    'bilateral',
    'crop_brain': True,
    'pretrained': True,
}

_CLS_DATA_CANDIDATES = [
    os.environ.get('ORACLE_CLS_DATA_ROOT', ''),
    '/kaggle/input/brain-tumor-mri-dataset',
    './brain-tumor-mri-dataset',
    './data/brain-tumor-mri-dataset',
]

def _first_existing_dir(cands):
    for c in cands:
        if c and Path(c).is_dir():
            return Path(c)
    return None

CLS_CFG = dict(CLS_CFG_DEFAULTS)
CLS_CFG.update({
    'data_root':      _first_existing_dir(_CLS_DATA_CANDIDATES),
    'batch_size':     32 if torch.cuda.is_available() else 8,
    'num_workers':    2 if torch.cuda.is_available() else 0,
    'val_frac':       0.15,
    'test_frac':      0.20,          # only used when split_mode == 'disjoint'
    # 'disjoint' : pool Training/+Testing/, cluster near-duplicates, and re-split
    #              so no duplicate group spans train/val/test. Honest, and LOWER.
    # 'official' : use the dataset's own Training//Testing/ split (leaky, comparable
    #              to published numbers).
    'split_mode':     'disjoint',
    'dhash_size':     16,            # 16x16 -> 256-bit hash. 8x8 (64-bit) is NOT
                                     # discriminative enough for brain MRI: duplicates
                                     # and distinct scans both land ~3 bits apart, so no
                                     # safe threshold exists and union-find collapses.
    'dup_hamming':    10,            # of 256 bits. Measured: duplicates <=7, distinct >=15.
    'train_repeats':  2,             # virtual dataset doubling (fresh augmentation each pass)
    'label_smooth':   0.05,
    'weight_decay':   1e-4,
    'class_weighting': 'inverse_freq',    # 'inverse_freq' | 'none'
    'ema_decay':      0.999,
    'epochs_head':    3,      # phase A — classifier head only
    'epochs_partial': 10,     # phase B — features[5:] + head
    'epochs_full':    37,     # phase C — everything, discriminative LR  (3+10+37 = 50)
    'patience':       8,      # early stop: val macro-F1 (EMA) stalls for N epochs
    'lr_head':        1e-3,
    'lr_partial':     3e-4,
    'lr_full_head':   3e-4,
    'lr_full_bb':     3e-5,
})

BEST_MODEL_PATH  = 'best_effnetb3_cls.pth'    # bare EMA state_dict — mirrors best_nnunet2d.pth
FINAL_MODEL_PATH = 'effnetb3_cls_final.pt'    # rich dict — mirrors pinn_tumor_growth.pt

# Since torch 1.6 `torch.save` writes a ZIP archive (magic bytes PK\x03\x04). Kaggle
# sniffs uploads by content, decides a .pth is an archive, and EXTRACTS it — so the
# checkpoint arrives in the model mount as a *directory* of unpacked internals
# (data.pkl, data/0, ...) and torch.load cannot read it.
#
# Writing the legacy (non-zip) pickle format sidesteps that entirely: the file is not
# an archive, so there is nothing for Kaggle to unpack. torch.load reads both formats,
# so nothing downstream changes. Set to False for the modern zip format (and then
# upload the checkpoints inside a .zip wrapper yourself).
CLS_CFG['kaggle_safe_save'] = True
SAVE_KW = {'_use_new_zipfile_serialization': False} if CLS_CFG['kaggle_safe_save'] else {}


def save_ckpt(obj, path):
    """torch.save + verify the file is not an archive Kaggle would auto-extract."""
    torch.save(obj, path, **SAVE_KW)
    if CLS_CFG['kaggle_safe_save'] and zipfile.is_zipfile(path):
        raise RuntimeError(
            f'{path} is still a ZIP archive — Kaggle will extract it into a folder. '
            'Legacy serialization did not take effect.')
    return path

DATA_AVAILABLE = CLS_CFG['data_root'] is not None
print('data_root :', CLS_CFG['data_root'])
if not DATA_AVAILABLE:
    print('  [!] Dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset on Kaggle,')
    print('      or set ORACLE_CLS_DATA_ROOT to a folder containing Training/ and Testing/.')
    print('      Definition-only cells (2, 4, 6, 7, 11, 12) still run.')

## 3. Dataset Discovery, De-duplication & Split

The dataset ships a `Training/` and `Testing/` split that is **not patient-disjoint** — the
same lesion appears in both, sometimes byte-identical, often as a rescale/recompress of the
same slice. Any metric computed across that boundary is partly a memorisation score.

`CLS_CFG['split_mode']` controls what we do about it:

- **`'disjoint'` (default)** — pool both folders, fingerprint every image with a 64-bit
  **dHash**, union-find them into near-duplicate groups, then split *by group* with
  `StratifiedGroupKFold` so no group spans train/val/test. Class balance is preserved.
- **`'official'`** — the shipped split, for comparability with published numbers.

**Expect `disjoint` to score several points LOWER.** That drop is leakage being removed, not
a regression — it is the more honest number.

On the threshold: near-duplicates measure ≤2 bits apart while genuinely different images sit
≥3, so `dup_hamming = 4` is deliberately a little loose. Over-merging is the *safe* direction
(it only makes the split more conservative); under-merging leaves leakage in. Union-find is
transitive though, so a too-large threshold can chain unrelated images into one giant group —
the cell prints the group-size distribution and warns if that happens.

In [ ]:
def _find_split(root: Path, name: str) -> Path:
    """Locate Training/ or Testing/ case-insensitively (mirrors vary)."""
    for p in root.iterdir():
        if p.is_dir() and p.name.lower() == name.lower():
            return p
    for p in root.rglob('*'):
        if p.is_dir() and p.name.lower() == name.lower():
            return p
    raise FileNotFoundError(f"No '{name}' folder under {root}")


def scan_split(split_dir: Path):
    """-> (files, labels, class_names) with class_names SORTED."""
    class_names = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    files, labels = [], []
    for ci, cn in enumerate(class_names):
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
            for f in sorted((split_dir / cn).glob(ext)):
                files.append(f); labels.append(ci)
    return files, np.array(labels), class_names


def dhash_bits(img_u8, size: int = 16) -> np.ndarray:
    """(size*size,) uint8 bit vector — a difference hash.

    Compares horizontally-adjacent pixels of a (size+1) x size thumbnail, so it is
    invariant to rescale, mild brightness shifts and JPEG recompression — the ways a
    duplicate reappears in an aggregated dataset — while staying sensitive to content.

    **size matters a great deal here.** At 8x8 (64 bits) every brain MRI hashes almost
    alike: measured on this kind of data, duplicates sit <=3 bits apart and *distinct*
    scans also reach 3, so the two distributions touch and no threshold separates them.
    Union-find then chains the whole dataset into one group. 16x16 (256 bits) separates
    cleanly: duplicates <=7, distinct >=15.
    """
    small = cv2.resize(img_u8, (size + 1, size), interpolation=cv2.INTER_AREA)
    return (small[:, 1:] > small[:, :-1]).astype(np.uint8).ravel()


def cluster_near_duplicates(bits: np.ndarray, max_hamming: int = 4,
                            chunk: int = 512) -> np.ndarray:
    """Union-Find over pairwise Hamming <= max_hamming. (N,64) bits -> (N,) group ids.

    Hamming from the binary identity  d = |a| + |b| - 2<a,b>, evaluated in row chunks
    so the N x N matrix is never materialised in full.
    """
    N = bits.shape[0]
    B = bits.astype(np.int16)
    S = B.sum(1)
    parent = np.arange(N)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)

    for i0 in range(0, N, chunk):
        i1 = min(i0 + chunk, N)
        D = S[i0:i1, None] + S[None, :] - 2 * (B[i0:i1] @ B.T)
        for r in range(i1 - i0):
            for j in np.nonzero(D[r] <= max_hamming)[0]:
                if j > i0 + r:
                    union(i0 + r, int(j))
    return np.array([find(i) for i in range(N)])


if not DATA_AVAILABLE:
    raise FileNotFoundError(
        'brain-tumor-mri-dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset '
        'on Kaggle, or set ORACLE_CLS_DATA_ROOT to a folder containing Training/ and Testing/, '
        'then re-run from here.')

TRAIN_DIR = _find_split(CLS_CFG['data_root'], 'Training')
TEST_DIR  = _find_split(CLS_CFG['data_root'], 'Testing')

off_tr_files, off_tr_labels, class_names  = scan_split(TRAIN_DIR)
off_te_files, off_te_labels, test_classes = scan_split(TEST_DIR)

# A silent reordering here would permute the checkpoint's output layer.
assert class_names == CLS_CFG['classes'], f'class order drift: {class_names}'
assert test_classes == class_names, f'test/train class mismatch: {test_classes}'

all_files  = list(off_tr_files) + list(off_te_files)
all_labels = np.concatenate([off_tr_labels, off_te_labels])
print(f'official Training/ : {len(off_tr_files):,}')
print(f'official Testing/  : {len(off_te_files):,}')
print(f'pooled             : {len(all_files):,}')

# ── Evidence for the leakage claim (exact matches only) ───────────────────
def _md5(p, n=1 << 16):
    h = hashlib.md5()
    with open(p, 'rb') as fh:
        for chunk in iter(lambda: fh.read(n), b''):
            h.update(chunk)
    return h.hexdigest()

_tr_h = {_md5(f) for f in off_tr_files}
_dupes = sum(1 for f in off_te_files if _md5(f) in _tr_h)
print(f'\nbyte-identical images shared across the official split: {_dupes} '
      f'({100*_dupes/max(len(off_te_files),1):.2f}% of Testing/)')

# ── Perceptual fingerprint + near-duplicate grouping ──────────────────────
print(f"\nfingerprinting {len(all_files):,} images (dHash)...")
_t0 = time.perf_counter()
_bits = np.stack([dhash_bits(cv2.imread(str(f), cv2.IMREAD_GRAYSCALE),
                                 size=CLS_CFG['dhash_size']) for f in all_files])
groups = cluster_near_duplicates(_bits, max_hamming=CLS_CFG['dup_hamming'])
_gsz = np.bincount(np.unique(groups, return_inverse=True)[1])
n_groups = len(_gsz)
print(f'  {time.perf_counter()-_t0:.1f}s  ->  {n_groups:,} groups from {len(all_files):,} images')
print(f'  group sizes: 1={int((_gsz==1).sum()):,}  2={int((_gsz==2).sum()):,}  '
      f'3-5={int(((_gsz>=3)&(_gsz<=5)).sum()):,}  >5={int((_gsz>5).sum()):,}  '
      f'largest={int(_gsz.max())}')
print(f'  near-duplicate images (in a group of >1): '
      f'{int(len(all_files) - n_groups):,} '
      f'({100*(len(all_files)-n_groups)/len(all_files):.1f}%)')

# Is the threshold actually separating on THIS data? Sample random pairs and look.
_rs = np.random.default_rng(SEED)
_ia = _rs.integers(0, len(all_files), 4000); _ib = _rs.integers(0, len(all_files), 4000)
_keep = _ia != _ib
_pd = (_bits[_ia[_keep]] != _bits[_ib[_keep]]).sum(1)
_nbits = _bits.shape[1]
print(f'  random-pair Hamming ({_nbits}-bit): min={_pd.min()} p1={int(np.percentile(_pd,1))} '
      f'median={int(np.median(_pd))}   threshold={CLS_CFG["dup_hamming"]}')
if np.percentile(_pd, 1) < CLS_CFG['dup_hamming'] * 1.5:
    print('  [!] threshold is close to the distance between UNRELATED images — expect over-merging.')

# Moderate groups are EXPECTED and wanted: adjacent slices from one patient are
# near-identical, and keeping them on one side of the split is the whole point.
# A runaway group means transitive union-find chained unrelated images together,
# which silently destroys stratification — so this is a hard stop, not a warning.
assert _gsz.max() <= 0.02 * len(all_files) and n_groups >= 0.30 * len(all_files), (
    f"near-duplicate grouping COLLAPSED: {n_groups} groups from {len(all_files)} images, "
    f"largest holds {int(_gsz.max())} ({100*_gsz.max()/len(all_files):.1f}%). Union-find is "
    f"transitive, so A~B~C merges A and C even when they differ. Fix: raise "
    f"CLS_CFG['dhash_size'] (now {CLS_CFG['dhash_size']}) and/or lower CLS_CFG['dup_hamming'] "
    f"(now {CLS_CFG['dup_hamming']} of {_nbits} bits), then re-run. "
    f"Groups of ~5-30 are normal (one patient's slice series); thousands are not. "
    f"To bypass entirely, set CLS_CFG['split_mode'] = 'official'.")

# ── Build the split ───────────────────────────────────────────────────────
if CLS_CFG['split_mode'] == 'disjoint':
    from sklearn.model_selection import StratifiedGroupKFold
    idx = np.arange(len(all_files))
    n_test = max(int(round(1 / CLS_CFG['test_frac'])), 2)
    sgk = StratifiedGroupKFold(n_splits=n_test, shuffle=True, random_state=SEED)
    rest_i, test_i = next(sgk.split(idx, all_labels, groups=groups))

    n_val = max(int(round(1 / CLS_CFG['val_frac'])), 2)
    sgk2 = StratifiedGroupKFold(n_splits=n_val, shuffle=True, random_state=SEED)
    tr_rel, va_rel = next(sgk2.split(rest_i, all_labels[rest_i], groups=groups[rest_i]))
    train_i, val_i = rest_i[tr_rel], rest_i[va_rel]

    tr_files   = [all_files[i] for i in train_i]; tr_labels   = all_labels[train_i]
    va_files   = [all_files[i] for i in val_i];   va_labels   = all_labels[val_i]
    test_files = [all_files[i] for i in test_i];  test_labels = all_labels[test_i]

    # The whole point — prove it.
    gt, gv, gs = set(groups[train_i]), set(groups[val_i]), set(groups[test_i])
    assert not (gt & gs), f'{len(gt & gs)} groups span train/test'
    assert not (gv & gs), f'{len(gv & gs)} groups span val/test'
    assert not (gt & gv), f'{len(gt & gv)} groups span train/val'
    print('\nsplit = DISJOINT (pooled + group-aware)')
    print('  group overlap train/val/test: NONE (asserted)')
else:
    tr_files, va_files, tr_labels, va_labels = train_test_split(
        off_tr_files, off_tr_labels, test_size=CLS_CFG['val_frac'],
        stratify=off_tr_labels, random_state=SEED)
    test_files, test_labels = off_te_files, off_te_labels
    print('\nsplit = OFFICIAL (shipped Training//Testing/ — leaky, see caveat 1)')

# A group-aware split can under-fill a fold when there are few groups relative to
# n_splits. Catch it here rather than as an IndexError further down the notebook.
for _nm, _fs, _ls in (('train', tr_files, tr_labels), ('val', va_files, va_labels),
                      ('test', test_files, test_labels)):
    assert len(_fs) > 0, (
        f"{_nm} split is EMPTY — {n_groups} near-duplicate groups is too few for "
        f"test_frac={CLS_CFG['test_frac']} / val_frac={CLS_CFG['val_frac']}.")
    # Missing classes are the failure that hides: val F1 reads a perfect 1.0 because
    # only one class is present, and roc_auc_score dies much later with an opaque
    # 'number of classes != columns in y_score'. Catch it here.
    _present = sorted(set(int(v) for v in _ls))
    assert _present == list(range(len(class_names))), (
        f"{_nm} split is missing classes: has {[class_names[i] for i in _present]}, "
        f"needs all {len(class_names)}. Group-aware splitting cannot stratify when "
        f"near-duplicate groups are too large — check the grouping report above, raise "
        f"CLS_CFG['dhash_size'] / lower CLS_CFG['dup_hamming'], or use split_mode='official'.")

print(f'  train {len(tr_files):,} | val {len(va_files):,} | test {len(test_files):,}')
counts = Counter(tr_labels.tolist())
print('\nclass balance (train):')
for i, c in enumerate(class_names):
    tr_n = int((tr_labels == i).sum()); te_n = int((test_labels == i).sum())
    print(f'  {c:12s} train={tr_n:5d}  val={int((va_labels==i).sum()):4d}  test={te_n:4d}')
imb = max(counts.values()) / max(min(counts.values()), 1)
print(f'imbalance max/min = {imb:.2f}x  '
      f'({"mild — class weights suffice" if imb < 1.5 else "consider resampling"})')

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
w = 0.27; xs = np.arange(len(class_names))
for k, (lab, y) in enumerate([('train', tr_labels), ('val', va_labels), ('test', test_labels)]):
    ax[0].bar(xs + (k - 1) * w, [int((y == i).sum()) for i in range(len(class_names))],
              width=w, label=lab)
ax[0].set_xticks(xs); ax[0].set_xticklabels(class_names, rotation=20)
ax[0].set_title(f"Class balance ({CLS_CFG['split_mode']} split)"); ax[0].legend(fontsize=8)
ax[1].hist(_gsz, bins=np.arange(1, min(_gsz.max(), 12) + 2) - 0.5, color='#7ec8c8')
ax[1].set_title('Near-duplicate group sizes'); ax[1].set_xlabel('images per group')
ax[1].set_yscale('log')
_sizes = [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE).shape for f in all_files[::60]]
ax[2].scatter([s[1] for s in _sizes], [s[0] for s in _sizes], s=6, alpha=0.35, color='#4a9eff')
ax[2].set_title('Image dimensions (every 60th)'); ax[2].set_xlabel('width'); ax[2].set_ylabel('height')
plt.tight_layout(); plt.show()

## 4. Preprocessing — Noise Reduction, Normalisation, Resizing

The pipeline stage the infographic calls *Image Preprocessing*, made explicit rather than
buried in a dataloader. Frozen chain, **identical here and in `full_pipeline_testing.ipynb`**:

```
read grayscale → crop_brain_region → resize(S,S) → denoise → /255 → 3ch → ImageNet norm → CHW
```

**Why bilateral?** These are already-compressed JPEGs, so the dominant artefact is 8×8 DCT
ringing at high-contrast edges — exactly what a range-weighted kernel suppresses while
leaving the edge intact. Gaussian is an unconditional low-pass and attenuates the
enhancing-rim / dural-tail high-frequency cues that separate glioma from meningioma. NLM
denoises best but costs ~40–120 ms/image, which at 5 712 images × ~21 epochs is hours of
pure CPU work; it stays reachable via `CLS_CFG['denoise'] = 'nlm'` and would need caching.
The cell below measures all three rather than asserting the choice.

In [ ]:
def crop_brain_region(img_u8: np.ndarray, thr: int = 10) -> np.ndarray:
    """Crop the black letterbox around the brain. Identity if nothing is found.

    This dataset's letterboxing varies wildly between source collections; cropping
    before resize keeps the brain at a consistent scale across images.
    """
    m = (img_u8 > thr).astype(np.uint8)
    if m.sum() == 0:
        return img_u8
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return img_u8
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = (stats[k, cv2.CC_STAT_LEFT], stats[k, cv2.CC_STAT_TOP],
                  stats[k, cv2.CC_STAT_WIDTH], stats[k, cv2.CC_STAT_HEIGHT])
    return img_u8[y:y + h, x:x + w] if w > 8 and h > 8 else img_u8


def denoise_u8(x: np.ndarray, mode: str) -> np.ndarray:
    if mode == 'bilateral':
        return cv2.bilateralFilter(x, d=5, sigmaColor=45, sigmaSpace=45)
    if mode == 'gaussian':
        return cv2.GaussianBlur(x, (0, 0), 1.0)
    if mode == 'nlm':
        return cv2.fastNlMeansDenoising(x, h=7)
    return x


def preprocess_u8(img_u8: np.ndarray, cfg: dict) -> np.ndarray:
    """uint8 grayscale -> preprocessed uint8 3-channel (S,S,3). Pre-normalisation."""
    S = int(cfg['img_size'])
    x = crop_brain_region(img_u8) if cfg.get('crop_brain', True) else img_u8
    x = cv2.resize(x, (S, S), interpolation=cv2.INTER_LINEAR)
    x = denoise_u8(x, cfg.get('denoise', 'none'))
    return np.stack([x, x, x], axis=-1)


def preprocess_cls(img_u8: np.ndarray, cfg: dict) -> torch.Tensor:
    """uint8 grayscale → (1,3,S,S) normalized tensor.

    Frozen chain (must stay identical to the training notebook, which asserts
    parity against its own val transform):
      crop_brain_region → resize(S,S) → bilateral → /255 → 3ch → ImageNet norm → CHW
    """
    S = int(cfg['img_size'])
    x = crop_brain_region(img_u8) if cfg.get('crop_brain', True) else img_u8
    x = cv2.resize(x, (S, S), interpolation=cv2.INTER_LINEAR)
    if cfg.get('denoise') == 'bilateral':
        x = cv2.bilateralFilter(x, d=5, sigmaColor=45, sigmaSpace=45)
    elif cfg.get('denoise') == 'nlm':
        x = cv2.fastNlMeansDenoising(x, h=7)
    f = x.astype(np.float32) / 255.0
    f = np.stack([f, f, f], axis=-1)
    f = (f - np.array(cfg['norm_mean'], np.float32)) / np.array(cfg['norm_std'], np.float32)
    return torch.from_numpy(f.transpose(2, 0, 1)).unsqueeze(0)


# ── Denoiser bake-off (demonstrate the choice, don't assert it) ───────────
# Sample one training image per class (tr_files/tr_labels come from section 3,
# and exist under both split modes).
_samples = []
for ci in range(len(CLS_CFG['classes'])):
    _hit = np.where(tr_labels == ci)[0]
    if len(_hit):
        _samples.append((CLS_CFG['classes'][ci], tr_files[int(_hit[0])]))

fig, axes = plt.subplots(len(_samples), 4, figsize=(11, 2.7 * len(_samples)))
modes = ['none', 'gaussian', 'bilateral', 'nlm']
timings = {m: [] for m in modes}
for r, (cn, fp) in enumerate(_samples):
    g = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(crop_brain_region(g), (CLS_CFG['img_size'],) * 2)
    for c, m in enumerate(modes):
        t0 = time.perf_counter(); out = denoise_u8(g, m); timings[m].append(time.perf_counter() - t0)
        axes[r, c].imshow(out, cmap='gray'); axes[r, c].axis('off')
        if r == 0: axes[r, c].set_title(m)
        if c == 0: axes[r, c].set_ylabel(cn)
    axes[r, 0].set_title(f'{cn}\n(original)' if r == 0 else cn, fontsize=9)
plt.tight_layout(); plt.show()
print('mean cost per image at %d^2:' % CLS_CFG['img_size'])
for m in modes:
    print(f'  {m:10s} {1000*np.mean(timings[m]):7.2f} ms')
print(f"\nselected: {CLS_CFG['denoise']}")

## 5. Dataset & Augmentation

**Virtual dataset expansion.** `CLS_CFG['train_repeats'] = 2` wraps the training set so each
epoch draws every image twice, with independent augmentation each time. Be clear about what
this does and does not do: it adds no new *information*, it doubles the number of augmented
views per LR-schedule step. Since the scheduler advances once per epoch, the practical effect
is twice the gradient steps at each point on the cosine curve — which is the useful part when
the raw set is small. Set it to 1 to disable.

**Stronger augmentation** to match the longer schedule: elastic and grid distortion (mild),
CLAHE, sharpen/blur, and wider affine. Two exclusions from the segmenter's list are kept
deliberately:

- **No `VerticalFlip`** — binary segmentation is orientation-agnostic, but a *classifier* must
  keep the brain's hard superior/inferior prior.
- **No `CoarseDropout`** — it can erase a small central pituitary lesion and flip the label.

**albumentations 2.x warning.** `nnUnet.ipynb` uses `ShiftScaleRotate`, `GaussNoise(var_limit=…)`
and `CoarseDropout(max_holes=…)`. In 2.x those kwargs were renamed — and critically
`GaussNoise(var_limit=…)` does **not** raise: it emits a `UserWarning` and silently drops the
argument, so the augmentation quietly becomes a no-op. The forms below are 2.x-native and the
assertion cell promotes warnings to errors so a silent drop fails loudly.

In [ ]:
def build_train_transform(cfg):
    """2.x-native forms only — see the section note on silently-dropped kwargs."""
    S, mean, std = cfg['img_size'], cfg['norm_mean'], cfg['norm_std']
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.08, 0.08),
                 rotate=(-15, 15), shear=(-6, 6), p=0.7),
        A.OneOf([
            A.ElasticTransform(alpha=25, sigma=6, p=1.0),
            A.GridDistortion(num_steps=5, distort_limit=0.2, p=1.0),
        ], p=0.25),
        A.RandomBrightnessContrast(0.25, 0.25, p=0.6),
        A.RandomGamma((80, 120), p=0.3),
        A.CLAHE(clip_limit=2.0, p=0.2),
        A.OneOf([A.Sharpen(p=1.0), A.GaussianBlur(blur_limit=(3, 5), p=1.0)], p=0.2),
        A.GaussNoise(p=0.25),                    # kwarg-free: survives the 2.x rename
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


def build_eval_transform(cfg):
    return A.Compose([A.Normalize(mean=cfg['norm_mean'], std=cfg['norm_std']), ToTensorV2()])


class BrainTumorDataset(Dataset):
    """Reads grayscale JPEG -> preprocess_u8 -> albumentations -> (CHW tensor, label).

    Geometry/denoise live in preprocess_u8 so train and inference share one chain;
    the transform only adds photometric/affine jitter and the ImageNet normalisation.
    """
    def __init__(self, files, labels, cfg, train: bool):
        self.files, self.labels, self.cfg = files, labels, cfg
        self.tf = build_train_transform(cfg) if train else build_eval_transform(cfg)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        g = cv2.imread(str(self.files[i]), cv2.IMREAD_GRAYSCALE)
        if g is None:
            g = np.zeros((self.cfg['img_size'],) * 2, np.uint8)
        x = preprocess_u8(g, self.cfg)
        return self.tf(image=x)['image'], int(self.labels[i])


class RepeatDataset(Dataset):
    """Virtually expand a dataset by `repeats`, re-drawing augmentation each pass.

    Adds no new information — it multiplies augmented views per LR-schedule step,
    since the scheduler advances once per epoch regardless of epoch length.
    """
    def __init__(self, base: Dataset, repeats: int = 2):
        self.base, self.repeats = base, max(int(repeats), 1)

    def __len__(self):
        return len(self.base) * self.repeats

    def __getitem__(self, i):
        return self.base[i % len(self.base)]


# Splits come from section 3 (group-disjoint or official, per CLS_CFG['split_mode']).
train_base = BrainTumorDataset(tr_files, tr_labels, CLS_CFG, train=True)
train_ds   = RepeatDataset(train_base, CLS_CFG['train_repeats'])
val_ds     = BrainTumorDataset(va_files, va_labels, CLS_CFG, train=False)
test_ds    = BrainTumorDataset(test_files, test_labels, CLS_CFG, train=False)

_dl = dict(batch_size=CLS_CFG['batch_size'], num_workers=CLS_CFG['num_workers'],
           pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_ds, shuffle=True,  drop_last=True,  **_dl)
val_loader   = DataLoader(val_ds,   shuffle=False, **_dl)
test_loader  = DataLoader(test_ds,  shuffle=False, **_dl)
print(f'train {len(train_base):,} x{CLS_CFG["train_repeats"]} = {len(train_ds):,} '
      f'({len(train_loader)} steps/epoch) | val {len(val_ds):,} | test {len(test_ds):,}')

# ── Preprocessing-parity assertion ────────────────────────────────────────
# The single guard that stops train-time and pipeline-time preprocessing drifting.
# Also promotes albumentations kwarg warnings to errors so a silently-ignored
# argument fails here rather than becoming a no-op augmentation.
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('error', UserWarning)
    _ = build_train_transform(CLS_CFG)
    _ = build_eval_transform(CLS_CFG)

_probe_file = (va_files or tr_files or test_files)[0]
_g = cv2.imread(str(_probe_file), cv2.IMREAD_GRAYSCALE)
_a = build_eval_transform(CLS_CFG)(image=preprocess_u8(_g, CLS_CFG))['image'].numpy()
_b = preprocess_cls(_g, CLS_CFG)[0].numpy()
assert np.allclose(_a, _b, atol=1e-5), f'preprocessing drift! max|d|={np.abs(_a-_b).max():.2e}'
print('preprocessing parity (dataset vs preprocess_cls): OK  '
      f'max|delta|={np.abs(_a - _b).max():.2e}')

# Augmentation sanity: same source image, four independent draws
_ex = preprocess_u8(cv2.imread(str(tr_files[0]), cv2.IMREAD_GRAYSCALE), CLS_CFG)
_tf = build_train_transform(CLS_CFG)
fig, ax = plt.subplots(1, 4, figsize=(11, 3))
for a in ax:
    v = _tf(image=_ex)['image'].numpy().transpose(1, 2, 0)
    v = np.clip(v * np.array(CLS_CFG['norm_std']) + np.array(CLS_CFG['norm_mean']), 0, 1)
    a.imshow(v); a.axis('off')
fig.suptitle('Four augmentation draws from one image'); plt.tight_layout(); plt.show()

## 6. Model — EfficientNet-B3 (ImageNet transfer)

The class body below is a **frozen contract**: `full_pipeline_testing.ipynb` re-declares it
character-for-character so `best_effnetb3_cls.pth` loads there with `strict=True`. Keys are
namespaced under `backbone.` and mirror torchvision's naming — switching to `timm` or
modifying the stem would break checkpoint loading downstream. Grayscale is replicated to 3
channels rather than surgically reducing the stem, for the same reason.

In [ ]:
class BrainTumorClassifier(nn.Module):
    """EfficientNet-B3 → 4-class brain-tumour classifier.

    state_dict keys are namespaced under `backbone.` so this class can be
    re-declared verbatim in full_pipeline_testing.ipynb and loaded strict=True.
    The constructor NEVER downloads weights — see load_imagenet_b3_weights().
    """
    def __init__(self, num_classes: int = 4, dropout: float = 0.3):
        super().__init__()
        from torchvision.models import efficientnet_b3
        self.backbone = efficientnet_b3(weights=None)
        in_f = self.backbone.classifier[1].in_features          # 1536
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout, inplace=True),
            nn.Linear(in_f, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)


B3_WEIGHT_GLOBS = [
    '/kaggle/input/**/efficientnet_b3*.pth',
    '/kaggle/input/**/efficientnet-b3*.pth',
    str(Path.home() / '.cache/torch/hub/checkpoints/efficientnet_b3*.pth'),
    '/kaggle/working/torch_hub/checkpoints/efficientnet_b3*.pth',
]


def load_imagenet_b3_weights(model) -> bool:
    """(1) torchvision download → (2) local glob → (3) random init.

    Returns the pretrained flag, which is STORED IN THE CHECKPOINT. Metrics from a
    random-init backbone look plausible but mean nothing, so this must never fail
    silently — section 13 refuses to declare success when this returns False.
    """
    sd = None
    try:
        from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
        sd = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1).state_dict()
        print('[weights] torchvision IMAGENET1K_V1 OK')
    except Exception as e:
        print(f'[weights] torchvision unavailable ({type(e).__name__}: {e})')
        for pat in B3_WEIGHT_GLOBS:
            hits = sorted(glob.glob(pat, recursive=True))
            if hits:
                sd = torch.load(hits[0], map_location='cpu')
                sd = sd.get('state_dict', sd)
                print(f'[weights] local: {hits[0]}')
                break
    if sd is None:
        print('!' * 70)
        print('[weights] NO IMAGENET WEIGHTS — TRAINING FROM RANDOM INIT.')
        print('[weights] Turn Kaggle Internet ON, or attach a dataset with efficientnet_b3*.pth.')
        print('[weights] Any metric produced this way is NOT comparable to a transfer run.')
        print('!' * 70)
        return False
    sd = {k: v for k, v in sd.items() if not k.startswith('classifier.')}
    miss, unexp = model.backbone.load_state_dict(sd, strict=False)
    print(f'[weights] loaded (missing={len(miss)} unexpected={len(unexp)}) '
          f'— missing should be exactly the 2 new classifier tensors')
    return True

## 7. Model Initialization, Loss, Optimizer, EMA

In [ ]:
class ModelEMA:
    """Exponential moving average of model weights.

    Maintains a shadow copy whose params evolve as:
        ema_param = decay * ema_param + (1 - decay) * model_param
    """

    def __init__(self, model, decay=0.999):
        unwrapped = model.module if isinstance(model, nn.DataParallel) else model
        self.module = copy.deepcopy(unwrapped).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        unwrapped = model.module if isinstance(model, nn.DataParallel) else model
        msd = unwrapped.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1.0 - self.decay)
            else:
                v.copy_(msd[k])


model = BrainTumorClassifier(num_classes=len(CLS_CFG['classes']),
                             dropout=CLS_CFG['dropout'])
CLS_CFG['pretrained'] = load_imagenet_b3_weights(model)
model = model.to(DEVICE)
ema = ModelEMA(model, decay=CLS_CFG['ema_decay'])

n_par = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_par/1e6:.2f} M   pretrained={CLS_CFG["pretrained"]}')

# ── Class weights ─────────────────────────────────────────────────────────
# Imbalance here is ~1.2x. Inverse-frequency weights are enough; a
# WeightedRandomSampler at this ratio only adds sampling variance without
# changing the effective class prior in any useful way.
if CLS_CFG['class_weighting'] == 'inverse_freq':
    _n = np.bincount(tr_labels, minlength=len(class_names)).astype(np.float64)
    _w = _n.sum() / (len(class_names) * np.maximum(_n, 1))
    class_weights = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
else:
    class_weights = None
print('class weights:', None if class_weights is None
      else {c: round(float(w), 3) for c, w in zip(class_names, class_weights)})

criterion = nn.CrossEntropyLoss(weight=class_weights,
                                label_smoothing=CLS_CFG['label_smooth'])
scaler = GradScaler(device='cuda', enabled=torch.cuda.is_available())


def set_trainable(model, phase: str):
    """Phase A: head only. B: features[5:] + head. C: everything."""
    for p in model.parameters():
        p.requires_grad_(phase == 'full')
    for p in model.backbone.classifier.parameters():
        p.requires_grad_(True)
    if phase == 'partial':
        for blk in model.backbone.features[5:]:
            for p in blk.parameters():
                p.requires_grad_(True)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [{phase}] trainable: {n/1e6:.2f} M / {n_par/1e6:.2f} M')


def freeze_bn(model):
    """Phase A only: keep ImageNet BatchNorm running stats intact while the head warms up
    on a tiny effective batch of gradient signal."""
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

## 8. Training — Three-Phase Transfer, 50 Epochs, Early Stopping

| Phase | Epochs | Trainable | LR |
| --- | --- | --- | --- |
| A · head warm-up | 3 | classifier head (BN frozen) | 1e-3 |
| B · partial unfreeze | 10 | `features[5:]` + head | 3e-4 |
| C · full fine-tune | 37 | everything | head 3e-4 / backbone 3e-5 |

**Early stopping** (`patience = 8`) watches val **macro-F1 of the EMA weights**. The counter
**resets at each phase boundary**: unfreezing changes the LR and the trainable set, so the
metric legitimately dips for a few epochs and a global counter would abort a healthy run.
Tripping it ends the *current phase* and moves on; in phase C it ends training. With 3 and 10
epoch phases, patience 8 only really bites in phase C — which is the intent.

Selection is on macro-F1, not accuracy, so the mild class imbalance cannot hide behind the
majority class. `CosineAnnealingLR` runs per phase rather than the repo's usual
`CosineAnnealingWarmRestarts` — warm restarts fight a staged freeze schedule.

> With `train_repeats = 2` this is up to 100 effective passes over the training set. On a
> Kaggle T4 at 300² expect roughly 2–4 minutes per epoch; early stopping usually cuts it well
> short of the 50-epoch ceiling. Watch the session time limit.

In [ ]:
@torch.no_grad()
def evaluate(net, loader, tta=False):
    net.eval()
    P, Y = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=torch.cuda.is_available()):
            logits = net(xb)
            if tta:                      # hflip only — vflip/rot90 are anatomically invalid
                logits = logits + net(torch.flip(xb, dims=[3]))
        P.append(torch.softmax(logits.float(), 1).cpu().numpy()); Y.append(yb.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    return P, Y, f1_score(Y, P.argmax(1), average='macro'), accuracy_score(Y, P.argmax(1))


def run_phase(phase, epochs, param_groups, history, best):
    """Train one phase. Returns True if early stopping fired.

    The patience counter is LOCAL to the phase: unfreezing changes both the LR and the
    trainable parameter set, so val F1 legitimately dips for a few epochs afterwards and
    a global counter would abort an otherwise healthy run.
    """
    set_trainable(model, phase)
    if phase == 'head':
        freeze_bn(model)
    opt = torch.optim.AdamW(param_groups, weight_decay=CLS_CFG['weight_decay'])
    sched = CosineAnnealingLR(opt, T_max=max(epochs, 1))
    phase_best, stale, nan_steps = -1.0, 0, 0

    for ep in range(epochs):
        model.train()
        if phase == 'head':
            freeze_bn(model)
        tot, seen, correct = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=torch.cuda.is_available()):
                out = model(xb); loss = criterion(out, yb)
            # A non-finite loss poisons every weight on the next step and every later
            # epoch reads NaN. Skip the step and abort if it keeps happening.
            if not torch.isfinite(loss):
                nan_steps += 1
                opt.zero_grad(set_to_none=True)
                if nan_steps <= 3:
                    print(f'    [!] non-finite loss at {phase} epoch {ep+1} — step skipped')
                if nan_steps > 20:
                    raise RuntimeError(
                        f'Loss went non-finite {nan_steps} times in phase {phase}. Usual causes: '
                        'a degenerate split (check the class counts in section 3), exploding '
                        'class weights from a near-empty class, or too high an LR.')
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update()
            ema.update(model)
            tot += loss.item() * yb.size(0); seen += yb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
        sched.step()

        if seen == 0:
            raise RuntimeError(f'No usable batches in {phase} epoch {ep+1} '
                               '(every step had a non-finite loss).')
        _, _, f1_raw, acc_raw = evaluate(model,      val_loader)
        _, _, f1_ema, acc_ema = evaluate(ema.module, val_loader)
        history.append({'phase': phase, 'epoch': ep + 1, 'train_loss': tot / seen,
                        'train_acc': correct / seen, 'val_f1': f1_raw, 'val_acc': acc_raw,
                        'val_f1_ema': f1_ema, 'val_acc_ema': acc_ema,
                        'lr': opt.param_groups[0]['lr']})

        star = ''
        if f1_ema > best['f1']:
            best.update(f1=f1_ema, phase=phase, epoch=ep + 1)
            save_ckpt(ema.module.state_dict(), BEST_MODEL_PATH)    # deployed = EMA
            star = '  <- best (saved)'
        # patience is tracked against the best seen WITHIN this phase
        if f1_ema > phase_best + 1e-5:
            phase_best, stale = f1_ema, 0
        else:
            stale += 1
        stale_note = '' if stale == 0 else f"  (stale {stale}/{CLS_CFG['patience']})"
        print(f'  [{phase} {ep+1}/{epochs}] loss={tot/seen:.4f} tr_acc={correct/seen:.4f} '
              f'| val F1 raw={f1_raw:.4f} ema={f1_ema:.4f} acc_ema={acc_ema:.4f}'
              f'{star}{stale_note}')

        if stale >= CLS_CFG['patience']:
            print(f"  early stop: no val macro-F1 gain in {CLS_CFG['patience']} epochs "
                  f"(phase best {phase_best:.4f})")
            return True
    return False


history, best = [], {'f1': -1.0, 'phase': None, 'epoch': -1}
t0 = time.time()
total_planned = (CLS_CFG['epochs_head'] + CLS_CFG['epochs_partial'] + CLS_CFG['epochs_full'])
print(f"schedule: {CLS_CFG['epochs_head']} + {CLS_CFG['epochs_partial']} + "
      f"{CLS_CFG['epochs_full']} = {total_planned} epochs max, "
      f"patience {CLS_CFG['patience']} (per phase), "
      f"{len(train_loader)} steps/epoch\n")

print('Phase A - head warm-up')
run_phase('head', CLS_CFG['epochs_head'],
          [{'params': model.backbone.classifier.parameters(), 'lr': CLS_CFG['lr_head']}],
          history, best)

print('Phase B - partial unfreeze')
run_phase('partial', CLS_CFG['epochs_partial'],
          [{'params': [p for p in model.parameters() if p.requires_grad],
            'lr': CLS_CFG['lr_partial']}], history, best)

print('Phase C - full fine-tune (discriminative LR)')
set_trainable(model, 'full')
stopped = run_phase('full', CLS_CFG['epochs_full'],
                    [{'params': model.backbone.features.parameters(), 'lr': CLS_CFG['lr_full_bb']},
                     {'params': model.backbone.classifier.parameters(), 'lr': CLS_CFG['lr_full_head']}],
                    history, best)

print(f"\nRan {len(history)} / {total_planned} epochs"
      f"{' (early stopped)' if stopped else ''}")
print(f"Best val macro-F1 = {best['f1']:.4f} "
      f"(phase {best['phase']}, epoch {best['epoch']}) -> {BEST_MODEL_PATH}")
print(f'Total training time: {(time.time()-t0)/60:.1f} min')

## 9. Training Visualization

In [ ]:
hist = pd.DataFrame(history)
hist['step'] = np.arange(1, len(hist) + 1)
_bounds = hist.groupby('phase', sort=False)['step'].max().tolist()[:-1]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(hist['step'], hist['train_loss'], color='#4a9eff'); ax[0].set_title('Train loss')
ax[1].plot(hist['step'], hist['val_f1'], label='raw', color='#4a9eff')
ax[1].plot(hist['step'], hist['val_f1_ema'], label='EMA', color='#7ec8c8', lw=2)
ax[1].set_title('Val macro-F1'); ax[1].legend()
ax[2].plot(hist['step'], hist['lr'], color='#ff6a6a'); ax[2].set_title('LR'); ax[2].set_yscale('log')
for a in ax:
    a.set_xlabel('epoch (cumulative)')
    for b in _bounds:
        a.axvline(b + 0.5, ls='--', lw=0.8, color='#888')
plt.tight_layout(); plt.show()
print(hist[['phase', 'epoch', 'train_loss', 'val_f1', 'val_f1_ema', 'val_acc_ema']]
      .to_string(index=False))

## 10. Test-Set Evaluation

`Testing/` is touched here and **only** here. Reported: accuracy, balanced accuracy,
macro-F1, per-class precision/recall/F1, confusion matrix (counts + row-normalised), and
one-vs-rest ROC-AUC. hflip TTA is reported **separately** rather than folded in.

Remember caveat 1: this split is not patient-disjoint from `Training/`, so these are upper
bounds, not generalisation estimates.

In [ ]:
# Evaluate the DEPLOYED weights (EMA), not the last SGD step.
eval_model = BrainTumorClassifier(num_classes=len(CLS_CFG['classes']),
                                  dropout=CLS_CFG['dropout']).to(DEVICE)
eval_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
eval_model.eval()

probs,     y_true, f1_m,   acc   = evaluate(eval_model, test_loader, tta=False)
probs_tta, _,      f1_tta, a_tta = evaluate(eval_model, test_loader, tta=True)
y_pred = probs.argmax(1)

bal = balanced_accuracy_score(y_true, y_pred)

# roc_auc_score infers the label space from y_true unless told otherwise, so a test
# split missing a class dies with 'number of classes != columns in y_score'. Passing
# labels= makes the intent explicit; the guard keeps a degenerate split from taking
# down the whole evaluation cell.
_all_labels = list(range(len(class_names)))
_missing = sorted(set(_all_labels) - set(int(v) for v in y_true))
if _missing:
    print('[!] test split is missing classes: '
          f"{[class_names[i] for i in _missing]} — ROC-AUC is undefined for them.")
    print('[!] This means the split is degenerate; treat every number below as invalid')
    print("[!] and re-check section 3's grouping report.")
    auc_macro = auc_w = float('nan')
else:
    auc_macro = roc_auc_score(y_true, probs, multi_class='ovr',
                              average='macro', labels=_all_labels)
    auc_w     = roc_auc_score(y_true, probs, multi_class='ovr',
                              average='weighted', labels=_all_labels)

print(f'accuracy          : {acc:.4f}')
print(f'balanced accuracy : {bal:.4f}')
print(f'macro F1          : {f1_m:.4f}')
print(f'ROC-AUC (ovr)     : macro={auc_macro:.4f}  weighted={auc_w:.4f}')
print(f'with hflip TTA    : accuracy={a_tta:.4f}  macro F1={f1_tta:.4f}')
print(f'\nmean max-softmax confidence: {probs.max(1).mean():.4f}')
print('NOTE: softmax confidence is NOT calibrated. It is exported to metrics.json and')
print('      shown in the public 3D viewer, so read it as a ranking score, not a probability.')

rep = classification_report(y_true, y_pred, labels=_all_labels,
                            target_names=class_names,
                            output_dict=True, zero_division=0)
print('\n' + pd.DataFrame(rep).T.round(4).to_string())

cm = confusion_matrix(y_true, y_pred, labels=_all_labels)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, M, t, fmt in ((ax[0], cm, 'Confusion matrix (counts)', 'd'),
                     (ax[1], cmn, 'Row-normalised (recall)', '.2f')):
    a.imshow(M, cmap='Blues'); a.set_xticks(range(len(class_names)))
    a.set_yticks(range(len(class_names)))
    a.set_xticklabels(class_names, rotation=45, ha='right'); a.set_yticklabels(class_names)
    a.set_xlabel('predicted'); a.set_ylabel('true'); a.set_title(t)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            a.text(j, i, format(M[i, j], fmt), ha='center', va='center',
                   color='white' if M[i, j] > M.max() * 0.55 else 'black', fontsize=9)
for i, c in enumerate(class_names):
    if len(set((y_true == i).astype(int))) < 2:      # class absent -> ROC undefined
        continue
    fpr, tpr, _ = roc_curve((y_true == i).astype(int), probs[:, i])
    ax[2].plot(fpr, tpr, lw=1.5,
               label=f'{c} (AUC={roc_auc_score((y_true==i).astype(int), probs[:,i]):.3f})')
ax[2].plot([0, 1], [0, 1], 'k--', lw=0.8); ax[2].set_title('One-vs-rest ROC')
ax[2].set_xlabel('FPR'); ax[2].set_ylabel('TPR'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

test_metrics = {'accuracy': float(acc), 'balanced_accuracy': float(bal),
                'macro_f1': float(f1_m), 'roc_auc_macro': float(auc_macro),
                'roc_auc_weighted': float(auc_w),
                'accuracy_tta': float(a_tta), 'macro_f1_tta': float(f1_tta),
                'per_class': {c: rep[c] for c in class_names},
                'confusion_matrix': cm.tolist(),
                'mean_max_softmax': float(probs.max(1).mean()),
                'test_split_duplicates_with_train': int(_dupes)}

## 11. Grad-CAM Explainability

Same hook machinery as `SegGradCAM` in `nnUnet.ipynb` §13 — only the scalar objective
changes: segmentation back-propagated `(prob * pred_mask).sum()`, here it is
`logits[0, class_idx]` (the raw logit, since softmax couples the classes).

Hook site is `backbone.features.8`, the final 1536-channel `Conv2dNormActivation` — the last
spatial tensor before global pooling, 10×10 at 300² input. `features.7` is a sharper but
noisier alternative, selectable via `CLS_CAM_LAYER`.

The **failure panel** at the end is the point of this section: correct-prediction CAMs are
reassuring, misclassification CAMs are informative.

In [ ]:
CLS_CAM_LAYER = 'backbone.features.8'   # final 1536-ch Conv2dNormActivation; 10×10 @ 300²

def get_cam_layer(model, key=CLS_CAM_LAYER):
    return dict(model.named_modules())[key]

def _disable_inplace(module):
    """Cheap insurance: in-place activations can interact badly with backward hooks
    on some torch builds. (Verified a no-op on torchvision 0.25 — gradients are
    identical either way — but it costs nothing and removes the failure mode.)"""
    for m in module.modules():
        if hasattr(m, 'inplace'):
            m.inplace = False

class ClsGradCAM:
    """Grad-CAM for a classifier — identical hook machinery to SegGradCAM
    (nnUnet.ipynb §13). Only the scalar objective differs: segmentation used
    (prob * pred_mask).sum(); here we use logits[0, class_idx]."""

    def __init__(self, model, target_layer):
        self.model = model
        self._acts = self._grads = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, _m, _i, out): self._acts = out.detach()
    def _save_grad(self, _m, _i, g):  self._grads = g[0].detach()

    def remove(self):
        self._fwd.remove(); self._bwd.remove()

    @torch.enable_grad()
    def __call__(self, image, class_idx=None):
        """image (1,3,H,W) → (cam H×W float32 in [0,1], class_idx, probs np[K])."""
        self.model.eval()
        self._acts = self._grads = None
        logits = self.model(image)
        probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        score = logits[0, class_idx]              # raw logit — softmax couples the classes
        self.model.zero_grad(set_to_none=True)
        score.backward()
        if self._grads is None or self._acts is None:
            return np.zeros(image.shape[-2:], np.float32), class_idx, probs
        weights = self._grads.mean(dim=[2, 3], keepdim=True)
        cam = torch.relu((weights * self._acts).sum(dim=1)).squeeze(0).cpu().numpy()
        cam = cv2.resize(cam, (image.shape[3], image.shape[2]), interpolation=cv2.INTER_LINEAR)
        lo, hi = float(cam.min()), float(cam.max())
        if hi > lo:
            cam = (cam - lo) / (hi - lo)
        return cam.astype(np.float32), class_idx, probs


ALPHA_BLEND = 0.55            # heatmap-over-image opacity (matches nnUnet.ipynb §13)

def cam_panel(indices, title):
    """indices -> rows of [input | Grad-CAM | overlay | class probabilities]."""
    _disable_inplace(eval_model)
    cam_engine = ClsGradCAM(eval_model, get_cam_layer(eval_model))
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(13, 3.1 * n))
    axes = np.atleast_2d(axes)
    for r, idx in enumerate(indices):
        g = cv2.imread(str(test_files[idx]), cv2.IMREAD_GRAYSCALE)
        x = preprocess_cls(g, CLS_CFG).to(DEVICE)
        cam, ci, pr = cam_engine(x)
        if not np.isfinite(cam).all() or float(cam.max()) - float(cam.min()) <= 0.0:
            print(f'[cam] WARNING: degenerate (flat) CAM at test index {idx}')
        base = cv2.resize(crop_brain_region(g), (CLS_CFG['img_size'],) * 2)
        rgb = np.stack([base] * 3, -1)
        heat = (plt.get_cmap('jet')(cam)[..., :3] * 255).astype(np.uint8)
        over = ((1 - ALPHA_BLEND) * rgb + ALPHA_BLEND * heat).astype(np.uint8)

        true_c, pred_c = class_names[int(test_labels[idx])], class_names[ci]
        ok = true_c == pred_c
        axes[r, 0].imshow(rgb); axes[r, 0].set_title(
            f'true: {true_c}', fontsize=9, color='green' if ok else 'red')
        axes[r, 1].imshow(cam, cmap='jet'); axes[r, 1].set_title('Grad-CAM', fontsize=9)
        axes[r, 2].imshow(over); axes[r, 2].set_title(
            f'pred: {pred_c} ({pr[ci]*100:.1f}%)', fontsize=9,
            color='green' if ok else 'red')
        for a in axes[r, :3]:
            a.axis('off')
        axes[r, 3].barh(class_names, pr,
                        color=['#7ec8c8' if i == ci else '#4a9eff' for i in range(len(class_names))])
        axes[r, 3].set_xlim(0, 1); axes[r, 3].tick_params(labelsize=8)
    cam_engine.remove()
    fig.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()


# One correct example per class
rng = np.random.default_rng(SEED)
correct_idx = []
for ci in range(len(class_names)):
    pool = np.where((y_true == ci) & (y_pred == ci))[0]
    if len(pool):
        correct_idx.append(int(rng.choice(pool)))
cam_panel(correct_idx, 'Grad-CAM — correct predictions (one per class)')

# The failure panel — what makes this explainability rather than decoration
wrong = np.where(y_true != y_pred)[0]
print(f'misclassified: {len(wrong)} / {len(y_true)}')
if len(wrong):
    cam_panel([int(i) for i in rng.choice(wrong, size=min(4, len(wrong)), replace=False)],
              'Grad-CAM — MISCLASSIFIED (where the model looked when it was wrong)')
else:
    print('No misclassifications on the test split — failure panel skipped.')

## 12. Save Checkpoints & Reload Helper

In [ ]:
def load_classifier(path, device=DEVICE):
    """Rebuild a BrainTumorClassifier from a checkpoint (self-contained, no globals).
    Returns (net, cfg, ckpt) — the same 3-tuple shape as load_pinn()."""
    obj = torch.load(path, map_location=device, weights_only=False)
    cfg_l = dict(CLS_CFG_DEFAULTS)
    if isinstance(obj, dict) and 'model_state' in obj:
        cfg_l.update(obj.get('config', {}))
        state, ckpt = (obj.get('ema_state') or obj['model_state']), obj
    else:
        state = obj.get('model_state_dict', obj) if isinstance(obj, dict) else obj
        ckpt = {'config': cfg_l}
    net = BrainTumorClassifier(num_classes=len(cfg_l['classes']), dropout=cfg_l['dropout']).to(device)
    net.load_state_dict(state)     # strict=True on purpose — a renamed layer must fail loud
    net.eval()
    return net, cfg_l, ckpt


# Only JSON-safe scalars go into `config` so the pipeline notebook can read it back.
_cfg_save = {k: (str(v) if isinstance(v, Path) else v)
             for k, v in CLS_CFG.items() if k != 'data_root'}

save_ckpt({
    'model_state': model.state_dict(),
    'ema_state':   ema.module.state_dict(),
    'config':      _cfg_save,
    'class_names': class_names,
    'test_metrics': test_metrics,
    'history':     history,
    'seed':        SEED,
    'torch_version': torch.__version__,
    'saved_at':    datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}, FINAL_MODEL_PATH)

for _p in (BEST_MODEL_PATH, FINAL_MODEL_PATH):
    _sz = Path(_p).stat().st_size / 1e6
    _zip = zipfile.is_zipfile(_p)
    print(f'  {_p:26s} {_sz:7.1f} MB   archive={_zip}   '
          f'{"[!] Kaggle WILL extract this" if _zip else "OK - Kaggle will leave it alone"}')

# Round-trip proof: the checkpoint must reload strict=True and reproduce its own logits.
_net, _cfg, _ck = load_classifier(FINAL_MODEL_PATH)
_probe = torch.randn(1, 3, CLS_CFG['img_size'], CLS_CFG['img_size'], device=DEVICE)
with torch.no_grad():
    _a = _net(_probe); _b = ema.module(_probe)
assert torch.allclose(_a, _b, atol=1e-4), 'reloaded EMA does not match in-memory EMA'
print(f'reload OK — classes={_cfg["classes"]}  img_size={_cfg["img_size"]}  '
      f'pretrained={_cfg["pretrained"]}')
# Belt and braces: a wrapper archive is safe to upload whatever the save format is,
# because Kaggle unpacks the WRAPPER and leaves the checkpoints inside it intact.
_bundle_dir = Path('/kaggle/working/ckpt_upload') if Path('/kaggle/working').is_dir() \
              else Path('./ckpt_upload')
_bundle_dir.mkdir(parents=True, exist_ok=True)
for _p in (BEST_MODEL_PATH, FINAL_MODEL_PATH):
    shutil.copy(_p, _bundle_dir / Path(_p).name)
_bundle = shutil.make_archive(str(_bundle_dir.parent / 'oracle_cls_ckpts'), 'zip',
                              root_dir=str(_bundle_dir))
print(f'\nupload bundle: {_bundle}  ({Path(_bundle).stat().st_size/1e6:.1f} MB)')

print('\nReload with:  net, cfg, ckpt = load_classifier(FINAL_MODEL_PATH)')
print('\nTo use these in full_pipeline_testing.ipynb Stage 0b:')
print('  1. Publish a NEW VERSION of the Kaggle model (the pipeline probes .../default/2')
print('     before .../default/1, so it picks up v2 automatically).')
print(f'  2. Upload either the two files above, or {Path(_bundle).name} if the UI')
print('     still unpacks them.')
print('  3. Confirm in the mount that they are FILES, not folders:')
print('       ls -la /kaggle/input/models/<you>/oracle-pipeline/pytorch/default/2/')
print('     A folder named *.pth means Kaggle extracted the checkpoint - re-upload zipped.')

## 13. Summary

In [ ]:
print('=' * 74)
print('  EfficientNet-B3 Brain Tumour Classification — Stage 0b')
print('=' * 74)
print(f"  classes        : {', '.join(class_names)}")
print(f"  input          : {CLS_CFG['img_size']}x{CLS_CFG['img_size']}, "
      f"{CLS_CFG['denoise']} denoise, brain-crop={CLS_CFG['crop_brain']}")
print(f"  params         : {n_par/1e6:.2f} M")
print(f"  split          : {CLS_CFG['split_mode']}"
      + ('  (pooled + near-duplicate-group disjoint)' if CLS_CFG['split_mode'] == 'disjoint'
         else '  (shipped split — LEAKY, see caveat 1)'))
print(f"  train set      : {len(tr_files):,} images x{CLS_CFG['train_repeats']} repeats")
print(f"  epochs run     : {len(history)} of {total_planned} max (patience {CLS_CFG['patience']})")
print(f"  deployed       : EMA weights (decay={CLS_CFG['ema_decay']}), selected on macro-F1")
print(f"  best val F1    : {best['f1']:.4f}  (phase {best['phase']}, epoch {best['epoch']})")
print('-' * 74)
print(f"  TEST accuracy  : {test_metrics['accuracy']:.4f}")
print(f"  TEST bal. acc  : {test_metrics['balanced_accuracy']:.4f}")
print(f"  TEST macro F1  : {test_metrics['macro_f1']:.4f}")
print(f"  TEST ROC-AUC   : {test_metrics['roc_auc_macro']:.4f} (ovr macro)")
print('=' * 74)

if not CLS_CFG['pretrained']:
    print()
    print('!' * 74)
    print('  RUN NOT VALID FOR REPORTING')
    print('  ImageNet weights were unavailable, so the backbone trained from random init.')
    print('  Turn Kaggle Internet ON (or attach efficientnet_b3 weights) and re-run before')
    print('  quoting ANY number above, and before publishing this checkpoint.')
    print('!' * 74)
else:
    print('  Transfer learning from ImageNet: OK')

print()
print('Caveats that travel with these numbers:')
if CLS_CFG['split_mode'] == 'disjoint':
    print('  1. Split is group-disjoint: images were pooled and re-split so no near-duplicate')
    print('     group spans train/val/test. These numbers are LOWER than the official-split')
    print('     figures and are the honest ones. (The shipped split shares')
    print(f"     {test_metrics['test_split_duplicates_with_train']} byte-identical images across its boundary.)")
else:
    print(f"  1. {test_metrics['test_split_duplicates_with_train']} test images are byte-identical to")
    print('     training images; the official split is not patient-disjoint. Upper bound, not')
    print('     a generalisation estimate. Set split_mode=\'disjoint\' for an honest number.')
print('  2. Softmax confidence is uncalibrated.')
print('  3. Applying this model to MU-Glioma-Post (full_pipeline_testing.ipynb Stage 0b) is')
print('     OUT-OF-DISTRIBUTION: that cohort is 100% glioma and post-operative, while this')
print('     dataset is pre-operative and multi-class. It is a wiring demo, not a diagnosis.')